# 1- Importing Libraries

In [16]:
! pip install pandas
! pip install numpy
! pip install scikit-learn
! pip install nltk
! pip install xgboost
! pip install seaborn
! pip install imbalanced-learn

In [17]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import nltk
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.naive_bayes import MultinomialNB
from nltk.tokenize import sent_tokenize,word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from imblearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix
nltk.download("stopwords")
nltk.download("punkt_tab")
lemmatizer = WordNetLemmatizer()
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/prabhsandhu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# 2- Loading Dataset

In [18]:
df = pd.read_csv("Reviews.csv")
df = df.sample(50000)

# 3- Understanding Dataset

In [3]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='str')

In [5]:
df.shape

(25000, 10)

In [6]:
df.nunique()
df["Score"].unique()

array([5, 4, 2, 3, 1])

In [7]:
df['HelpfulnessNumerator']

187136    0
357882    1
261049    1
258164    2
241834    2
         ..
479652    0
285216    0
452647    1
345481    3
486887    0
Name: HelpfulnessNumerator, Length: 25000, dtype: int64

# 4- Preprocessing the Dataset

In [6]:
df.isnull().sum()

Id                        0
ProductId                 0
UserId                    0
ProfileName               0
HelpfulnessNumerator      0
HelpfulnessDenominator    0
Score                     0
Time                      0
Summary                   1
Text                      0
dtype: int64

In [4]:
df.dropna(inplace=True)

In [5]:
df.duplicated().sum()

np.int64(0)

In [19]:
df.drop(['Id','UserId','Time','Summary','HelpfulnessNumerator','HelpfulnessDenominator','ProfileName'],axis=1,inplace=True)

In [20]:
df["Sentiment"] = df["Score"].apply(lambda x:"Negative" if x<=2 else ("Neutral" if x==3 else "Positive"))

In [21]:
df.drop("Score",axis=1,inplace=True)

# 5- Text Preprocessing Using NLP

In [22]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))  # load ONCE

text_preprocessed = []

for text in tqdm(df["Text"].astype(str)):  # ensure all rows are strings
    text = text.lower()
    text = re.sub(r'[^A-Za-z0-9]', ' ', text)
    
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words and len(word) >= 2]
    
    text_preprocessed.append(" ".join(words))

100%|██████████| 50000/50000 [00:17<00:00, 2843.10it/s]


# 6- Training Machine Learning Models Using Count Vectorizer as Technique

In [23]:
X = text_preprocessed
y = df["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

pipe = ImbPipeline([
    ('CountVec', CountVectorizer()),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=1000
    ))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

In [24]:
print(accuracy_score(y_test,y_pred)*100)
print(f1_score(y_test,y_pred,average="weighted")*100)
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

80.39333333333335
81.70439110799884
              precision    recall  f1-score   support

    Negative       0.60      0.67      0.63      2185
     Neutral       0.27      0.41      0.33      1111
    Positive       0.93      0.87      0.90     11704

    accuracy                           0.80     15000
   macro avg       0.60      0.65      0.62     15000
weighted avg       0.83      0.80      0.82     15000

[[ 1471   348   366]
 [  290   455   366]
 [  704   867 10133]]


# 7- Training Machine Learning Models Using Tf-Idf as Technique

In [25]:
X = text_preprocessed
y = df["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=15000)),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=5000
    ))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

In [26]:
print(accuracy_score(y_test,y_pred)*100)
print(f1_score(y_test,y_pred,average="weighted")*100)
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

80.44
82.3091402734274
              precision    recall  f1-score   support

    Negative       0.61      0.73      0.66      2185
     Neutral       0.28      0.50      0.36      1111
    Positive       0.95      0.85      0.90     11704

    accuracy                           0.80     15000
   macro avg       0.62      0.69      0.64     15000
weighted avg       0.85      0.80      0.82     15000

[[1585  358  242]
 [ 287  559  265]
 [ 715 1067 9922]]
